# 5. Сохраняемый отклик и новые положения ОМ

Одна транспортная задача используется для разных положений источника и
приёмников. В этой работе измеряем подготовку отдельно от применения и
выясняем, почему гладкая приёмка не требует сотен выходных гармоник.

In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium, SolverSettings, PointGreenSolver

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())

## 5.1. Открытый угловой индекс

В кэше хранятся
$$\mathcal R_\ell^{(p)}(r,\omega)=\frac{i^\ell}{2\pi^2}\sqrt{\frac{2\ell+1}{2}}
\int k^2j_\ell(kr)h_\ell^{(p)}(k,\omega)dk.$$
Индекс $p$ здесь обозначает первый порядок конечного $L$ или остаток $\ge2$.
Он не является номером базисной функции.

Для изотропной вспышки и осесимметричной приёмки
$$K_A=\sum_\ell\frac{\alpha_\ell}{4\pi}\mathcal R_\ell P_\ell(\xi),
\quad\alpha_\ell=2\pi\int A(x)P_\ell(x)dx,\quad\xi=-\mathbf n\cdot\widehat{\mathbf r}.$$
$\mathbf n$ смотрит из фотокатода наружу; $\xi=1$ означает, что он смотрит
на вспышку. Баллистический вклад добавляется аналитически.

При $A=(1+x)/2$ нужны только $\ell=0,1$:
$K_A=\mathcal R_0/2+\xi\mathcal R_1/6$.
Система рассеяния при этом по-прежнему имеет размер $L+1$.

In [ ]:
from lighthit.cache import CacheGrid,ResponseCache,BandedResponseCache
from lighthit.cache import acceptance_coefficients,hemispherical_acceptance
settings=SolverSettings(32,1,6.,.025,8)
grid=CacheGrid.geometric(10.,80.,24,[0.])
t0=time.perf_counter();cache=ResponseCache.build(medium,settings,grid);build=time.perf_counter()-t0
print('Build [s]:',build,'moment bytes:',cache.moments.nbytes)
alpha=acceptance_coefficients(hemispherical_acceptance,4)
print('alpha =',alpha)
np.testing.assert_allclose(alpha[:2],[2*np.pi,2*np.pi/3],rtol=1e-12)

In [ ]:
rng=np.random.default_rng(31)
vec=rng.normal(size=(576,3));vec/=np.linalg.norm(vec,axis=1)[:,None]
positions=vec*rng.uniform(20,65,size=576)[:,None]
receiver_axis=np.array([0.,0.,-1.]);banded=BandedResponseCache([cache])
def query(source):
    return banded.charge_for_modules(positions-source,receiver_axis,alpha,
                                    acceptance=hemispherical_acceptance)
t0=time.perf_counter();charge=query(np.zeros(3));first=time.perf_counter()-t0
samples=[]
for _ in range(5):
    t0=time.perf_counter();moved=query(np.array([2.,-1.,3.]));samples.append(time.perf_counter()-t0)
print('First query [s]:',first,'repeat/new-pose median [s]:',np.median(samples))
print('Source move changes charges:',np.max(np.abs(moved-charge)))
fig,ax=plt.subplots();ax.scatter(np.linalg.norm(positions,axis=1),charge.sum(1),s=8);ax.set(xlabel='r [m]',ylabel='charge [m^-2]',yscale='log');plt.show()

## 5.2. Проверки, которые должны останавливать расчёт

В кэше должны совпадать среда и частотная сетка всех диапазонов. Нельзя
молча отбрасывать ненулевые коэффициенты приёмки выше сохранённого $J$.
Опция `exact_first_order=True` разрешена только для постоянной приёмки:
после рассеяния свет не приходит из одного геометрического направления.

In [ ]:
try:
    cache.acceptance_charge([20.],[-1.],alpha,exact_first_order=True)
except ValueError as exc:
    print('Expected guard:',exc)
else:
    raise AssertionError('Directional exact-first shortcut must be rejected')
# Compare necessary low moments with an expensive high-J build at the same radii.
high=ResponseCache.build(medium,SolverSettings(32,80,6.,.025,8),
                         CacheGrid(np.array([20.,30.]),np.array([0.])))
low=ResponseCache.build(medium,settings,high.grid)
np.testing.assert_allclose(high.moments[:,:,:2],low.moments,rtol=1e-10,atol=1e-15)
print('Low moments agree; L is unchanged.')

## 5.3. Для временного кэша требуется отдельная проверка

Разрешена опция `radial_phase='flight'`: перед радиальной интерполяцией
снимается $e^{i\omega r/v}$, после неё фаза возвращается. Она не меняет
значения на сетке и не включает аппаратное размытие. Это средство улучшения
интерполяции, а не готовая оценка её ошибки: сравните промежуточные радиусы
с прямым решателем.

## Задания

Сделать случайные позы **после** построения таблицы; измерить медиа́ну и 95-й
перцентиль. Разделить время подготовки, загрузки и запроса. Сравнить
обе радиальные интерполяции на ненулевых частотах. Продемонстрировать
контрпример к ошибочному умножению первого порядка на $A(\xi)$.